# اليوم الثالث — مختبر 6: التقييم وتحليل الأخطاء
## Day 3 — Lab 6: Evaluation & Error Analysis

**المدربة / Instructor:** ميعاد المري — Meaad Al-Marri  
**المسار:** 🟢 Core → 🔵 Explore → 🟣 Distinction

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/almiyead-rgb/bayan-applied-nlp-course/blob/develop/notebooks/07_evaluation_error_analysis.ipynb)

**الهدف:** الانتقال من رقم عام إلى دليل: Macro-F1 + bootstrap CI + paired comparison + slices + behavioural tests + taxonomy + إصلاحات مرتبة.

**Goal:** produce an honest evaluation report that shows uncertainty and actionable failure modes.


## مهم: ما مصدر التنبؤات؟ | Evidence provenance

ملف `bayan_day3_predictions.csv` هو `COURSE_FIXTURE`: تنبؤات تعليمية مكتوبة خصيصًا لتعلم القياس، وليست نتائج نموذج مخفي ولا benchmark. عند استخدام مشروعك، استبدلها بتنبؤات validation فعلية مع الحفاظ على الأعمدة.

لا نستخدم frozen test لصناعة taxonomy أو اختيار الإصلاح؛ نستخدم validation حتى يظل test تقييمًا نهائيًا.


In [ ]:
import importlib.metadata
import subprocess
import sys

REQUIRED = {"scikit-learn": "1.9.0"}
try:
    current = importlib.metadata.version("scikit-learn")
except importlib.metadata.PackageNotFoundError:
    current = None
if current != REQUIRED["scikit-learn"]:
    subprocess.check_call([
        sys.executable, "-m", "pip", "install", "--quiet", "scikit-learn==1.9.0"
    ])
assert importlib.metadata.version("scikit-learn") == "1.9.0"
print("SETUP=PASS", importlib.metadata.version("scikit-learn"))


In [ ]:
import csv
import io
import json
import urllib.request
from collections import Counter
from pathlib import Path

import numpy as np
from sklearn.metrics import f1_score

DATA_KIND = "COURSE_FIXTURE"
print("IMPORTS=PASS")


In [ ]:
DATA_URL = "https://raw.githubusercontent.com/almiyead-rgb/bayan-applied-nlp-course/develop/data/sample/bayan_day3_predictions.csv"
FALLBACK_ROWS = [{'example_id': 'EV-001', 'split': 'validation', 'language': 'ar', 'variant': 'Gulf', 'length_bucket': 'short', 'topic': 'digital_service', 'prediction_a': 'permit', 'prediction_b': 'digital_service', 'text': 'ما وصلني رمز التحقق'}, {'example_id': 'EV-002', 'split': 'validation', 'language': 'ar', 'variant': 'Gulf', 'length_bucket': 'long', 'topic': 'health', 'prediction_a': 'health', 'prediction_b': 'health', 'text': 'أبي أغير موعد العيادة لأن الوقت الحالي ما يناسبني'}, {'example_id': 'EV-003', 'split': 'validation', 'language': 'ar', 'variant': 'Gulf', 'length_bucket': 'short', 'topic': 'permit', 'prediction_a': 'permit', 'prediction_b': 'permit', 'text': 'طلبي واقف بالمراجعة'}, {'example_id': 'EV-004', 'split': 'validation', 'language': 'ar', 'variant': 'Gulf', 'length_bucket': 'long', 'topic': 'transport', 'prediction_a': 'transport', 'prediction_b': 'transport', 'text': 'الباص تأخر علينا أكثر من ساعة وما وصل إشعار'}, {'example_id': 'EV-005', 'split': 'validation', 'language': 'ar', 'variant': 'Gulf', 'length_bucket': 'short', 'topic': 'digital_service', 'prediction_a': 'digital_service', 'prediction_b': 'digital_service', 'text': 'التطبيق يعلق عند الدخول'}, {'example_id': 'EV-006', 'split': 'validation', 'language': 'ar', 'variant': 'Gulf', 'length_bucket': 'long', 'topic': 'health', 'prediction_a': 'transport', 'prediction_b': 'transport', 'text': 'النتيجة للحين ما ظهرت بالتطبيق رغم مرور يومين'}, {'example_id': 'EV-007', 'split': 'validation', 'language': 'ar', 'variant': 'Gulf', 'length_bucket': 'short', 'topic': 'permit', 'prediction_a': 'digital_service', 'prediction_b': 'digital_service', 'text': 'المرفق انرفض'}, {'example_id': 'EV-008', 'split': 'validation', 'language': 'ar', 'variant': 'Gulf', 'length_bucket': 'long', 'topic': 'transport', 'prediction_a': 'transport', 'prediction_b': 'transport', 'text': 'المسار الجديد مو موجود في الخريطة بعد التحديث'}, {'example_id': 'EV-009', 'split': 'validation', 'language': 'ar', 'variant': 'Gulf', 'length_bucket': 'short', 'topic': 'digital_service', 'prediction_a': 'digital_service', 'prediction_b': 'digital_service', 'text': 'الكود ما وصل'}, {'example_id': 'EV-010', 'split': 'validation', 'language': 'ar', 'variant': 'Gulf', 'length_bucket': 'long', 'topic': 'health', 'prediction_a': 'health', 'prediction_b': 'digital_service', 'text': 'أحتاج أجدد الوصفة من التطبيق لكن الزر ما يشتغل'}, {'example_id': 'EV-011', 'split': 'validation', 'language': 'ar', 'variant': 'Gulf', 'length_bucket': 'short', 'topic': 'permit', 'prediction_a': 'permit', 'prediction_b': 'permit', 'text': 'انخصمت الرسوم مرتين'}, {'example_id': 'EV-012', 'split': 'validation', 'language': 'ar', 'variant': 'Gulf', 'length_bucket': 'long', 'topic': 'transport', 'prediction_a': 'digital_service', 'prediction_b': 'digital_service', 'text': 'وين موقف الباص الجديد في الحي؟'}, {'example_id': 'EV-013', 'split': 'validation', 'language': 'ar', 'variant': 'MSA', 'length_bucket': 'short', 'topic': 'digital_service', 'prediction_a': 'digital_service', 'prediction_b': 'digital_service', 'text': 'تعذر الدخول إلى البوابة'}, {'example_id': 'EV-014', 'split': 'validation', 'language': 'ar', 'variant': 'MSA', 'length_bucket': 'long', 'topic': 'health', 'prediction_a': 'health', 'prediction_b': 'health', 'text': 'لا تتوفر مواعيد مناسبة في العيادة المطلوبة'}, {'example_id': 'EV-015', 'split': 'validation', 'language': 'ar', 'variant': 'MSA', 'length_bucket': 'short', 'topic': 'permit', 'prediction_a': 'permit', 'prediction_b': 'permit', 'text': 'رُفض المستند المرفق'}, {'example_id': 'EV-016', 'split': 'validation', 'language': 'ar', 'variant': 'MSA', 'length_bucket': 'long', 'topic': 'transport', 'prediction_a': 'transport', 'prediction_b': 'transport', 'text': 'تأخرت الحافلة عن الموعد المحدد بنصف ساعة'}, {'example_id': 'EV-017', 'split': 'validation', 'language': 'ar', 'variant': 'MSA', 'length_bucket': 'short', 'topic': 'digital_service', 'prediction_a': 'permit', 'prediction_b': 'digital_service', 'text': 'يفشل رفع الملف'}, {'example_id': 'EV-018', 'split': 'validation', 'language': 'ar', 'variant': 'MSA', 'length_bucket': 'long', 'topic': 'health', 'prediction_a': 'health', 'prediction_b': 'health', 'text': 'لم تظهر نتيجة الفحص في الملف الصحي الإلكتروني'}, {'example_id': 'EV-019', 'split': 'validation', 'language': 'ar', 'variant': 'MSA', 'length_bucket': 'short', 'topic': 'permit', 'prediction_a': 'digital_service', 'prediction_b': 'digital_service', 'text': 'حالة التصريح معلقة'}, {'example_id': 'EV-020', 'split': 'validation', 'language': 'ar', 'variant': 'MSA', 'length_bucket': 'long', 'topic': 'transport', 'prediction_a': 'transport', 'prediction_b': 'digital_service', 'text': 'لا يظهر مسار الحافلة في التطبيق'}, {'example_id': 'EV-021', 'split': 'validation', 'language': 'ar', 'variant': 'MSA', 'length_bucket': 'short', 'topic': 'digital_service', 'prediction_a': 'digital_service', 'prediction_b': 'digital_service', 'text': 'لم يصل رمز التحقق'}, {'example_id': 'EV-022', 'split': 'validation', 'language': 'ar', 'variant': 'MSA', 'length_bucket': 'long', 'topic': 'health', 'prediction_a': 'health', 'prediction_b': 'health', 'text': 'تعذر تجديد الوصفة الطبية عبر الخدمة الإلكترونية'}, {'example_id': 'EV-023', 'split': 'validation', 'language': 'ar', 'variant': 'MSA', 'length_bucket': 'short', 'topic': 'permit', 'prediction_a': 'permit', 'prediction_b': 'permit', 'text': 'تم خصم الرسم مرتين'}, {'example_id': 'EV-024', 'split': 'validation', 'language': 'ar', 'variant': 'MSA', 'length_bucket': 'long', 'topic': 'transport', 'prediction_a': 'transport', 'prediction_b': 'transport', 'text': 'أرغب في معرفة موقع أقرب محطة للحافلات'}, {'example_id': 'EV-025', 'split': 'validation', 'language': 'en', 'variant': 'English', 'length_bucket': 'short', 'topic': 'digital_service', 'prediction_a': 'digital_service', 'prediction_b': 'digital_service', 'text': 'The sign in code did not arrive'}, {'example_id': 'EV-026', 'split': 'validation', 'language': 'en', 'variant': 'English', 'length_bucket': 'long', 'topic': 'health', 'prediction_a': 'health', 'prediction_b': 'health', 'text': 'I need to move my clinic appointment to another day'}, {'example_id': 'EV-027', 'split': 'validation', 'language': 'en', 'variant': 'English', 'length_bucket': 'short', 'topic': 'permit', 'prediction_a': 'permit', 'prediction_b': 'permit', 'text': 'My permit document was rejected'}, {'example_id': 'EV-028', 'split': 'validation', 'language': 'en', 'variant': 'English', 'length_bucket': 'long', 'topic': 'transport', 'prediction_a': 'transport', 'prediction_b': 'transport', 'text': 'The bus arrived forty minutes after the scheduled time'}, {'example_id': 'EV-029', 'split': 'validation', 'language': 'en', 'variant': 'English', 'length_bucket': 'short', 'topic': 'digital_service', 'prediction_a': 'digital_service', 'prediction_b': 'digital_service', 'text': 'The portal freezes'}, {'example_id': 'EV-030', 'split': 'validation', 'language': 'en', 'variant': 'English', 'length_bucket': 'long', 'topic': 'health', 'prediction_a': 'transport', 'prediction_b': 'transport', 'text': 'My laboratory result is still missing from the health record'}, {'example_id': 'EV-031', 'split': 'validation', 'language': 'en', 'variant': 'English', 'length_bucket': 'short', 'topic': 'permit', 'prediction_a': 'digital_service', 'prediction_b': 'digital_service', 'text': 'The request is stuck'}, {'example_id': 'EV-032', 'split': 'validation', 'language': 'en', 'variant': 'English', 'length_bucket': 'long', 'topic': 'transport', 'prediction_a': 'transport', 'prediction_b': 'transport', 'text': 'The new bus route is missing from the mobile map'}, {'example_id': 'EV-033', 'split': 'validation', 'language': 'en', 'variant': 'English', 'length_bucket': 'short', 'topic': 'digital_service', 'prediction_a': 'digital_service', 'prediction_b': 'digital_service', 'text': 'PDF upload failed'}, {'example_id': 'EV-034', 'split': 'validation', 'language': 'en', 'variant': 'English', 'length_bucket': 'long', 'topic': 'health', 'prediction_a': 'health', 'prediction_b': 'health', 'text': 'The prescription renewal button does not work in the application'}, {'example_id': 'EV-035', 'split': 'validation', 'language': 'en', 'variant': 'English', 'length_bucket': 'short', 'topic': 'permit', 'prediction_a': 'permit', 'prediction_b': 'permit', 'text': 'I was charged twice'}, {'example_id': 'EV-036', 'split': 'validation', 'language': 'en', 'variant': 'English', 'length_bucket': 'long', 'topic': 'transport', 'prediction_a': 'transport', 'prediction_b': 'transport', 'text': 'I cannot find the neighbourhood bus stop'}]

try:
    with urllib.request.urlopen(DATA_URL, timeout=15) as response:
        rows = list(csv.DictReader(io.StringIO(response.read().decode("utf-8"))))
    data_source = "github"
except Exception as exc:
    rows = FALLBACK_ROWS
    data_source = f"embedded_fallback:{type(exc).__name__}"

required = {"example_id", "split", "language", "variant", "length_bucket", "topic", "prediction_a", "prediction_b", "text"}
assert len(rows) == 36 and required <= set(rows[0])
assert {row["split"] for row in rows} == {"validation"}
print({"data_kind": DATA_KIND, "source": data_source, "rows": len(rows)})


## 1) Macro-F1 و95% bootstrap CI

Macro-F1 يعطي كل فئة وزنًا متساويًا. الـbootstrap يعيد أخذ أمثلة **paired** مع الاستبدال ليظهر عدم اليقين في هذه العينة الصغيرة. CI ليست وعدًا بأداء الإنتاج، ولا يعالج رفع `n_boot` نقص البيانات.


In [ ]:
def macro_f1(y_true, y_pred):
    return float(f1_score(y_true, y_pred, average="macro", zero_division=0))


def bootstrap_ci(y_true, y_pred, metric_fn, n_boot=1000, alpha=0.05, seed=42):
    truth = np.asarray(y_true, dtype=object)
    prediction = np.asarray(y_pred, dtype=object)
    if len(truth) == 0 or len(truth) != len(prediction):
        raise ValueError("paired non-empty arrays are required")
    rng = np.random.default_rng(seed)
    values = []
    for _ in range(n_boot):
        indexes = rng.integers(0, len(truth), len(truth))
        values.append(metric_fn(truth[indexes], prediction[indexes]))
    low, high = np.percentile(values, [100 * alpha / 2, 100 * (1 - alpha / 2)])
    return {
        "estimate": metric_fn(truth, prediction),
        "ci_low": float(low),
        "ci_high": float(high),
        "n_boot": n_boot,
    }

y_true = [row["topic"] for row in rows]
interval_a = bootstrap_ci(y_true, [row["prediction_a"] for row in rows], macro_f1)
interval_b = bootstrap_ci(y_true, [row["prediction_b"] for row in rows], macro_f1)
print("COURSE_FIXTURE A", interval_a)
print("COURSE_FIXTURE B", interval_b)


## 2) مقارنة زوجية عادلة | Paired bootstrap B − A

يجب أن يستخدم الإصداران الأمثلة نفسها في كل resample. إذا شملت CI الصفر، لا نقول إن B أفضل أو أسوأ اتجاهيًا؛ نقول إن العينة لا تدعم هذا الادعاء.


In [ ]:
def paired_bootstrap_difference(y_true, prediction_a, prediction_b, metric_fn, n_boot=1000, alpha=0.05, seed=42):
    truth = np.asarray(y_true, dtype=object)
    first = np.asarray(prediction_a, dtype=object)
    second = np.asarray(prediction_b, dtype=object)
    if len(truth) == 0 or not (len(truth) == len(first) == len(second)):
        raise ValueError("three paired non-empty arrays are required")
    rng = np.random.default_rng(seed)
    differences = []
    for _ in range(n_boot):
        indexes = rng.integers(0, len(truth), len(truth))
        differences.append(metric_fn(truth[indexes], second[indexes]) - metric_fn(truth[indexes], first[indexes]))
    low, high = np.percentile(differences, [100 * alpha / 2, 100 * (1 - alpha / 2)])
    observed = metric_fn(truth, second) - metric_fn(truth, first)
    return {
        "difference_b_minus_a": float(observed),
        "ci_low": float(low),
        "ci_high": float(high),
        "supports_directional_claim": bool(low > 0 or high < 0),
    }

paired_result = paired_bootstrap_difference(
    y_true,
    [row["prediction_a"] for row in rows],
    [row["prediction_b"] for row in rows],
    macro_f1,
)
verdict = (
    "The interval excludes zero; report the observed direction with scope and limitations."
    if paired_result["supports_directional_claim"]
    else "The interval includes zero; this fixture does not support a directional superiority claim."
)
print("COURSE_FIXTURE paired B-A", paired_result)
print("VERDICT:", verdict)


## 3) Sliced evaluation: المتوسط لا يكفي

نقرأ الأداء حسب `language` و`variant` و`length_bucket`. نستخدم حد مراجعة تعليميًا قدره 15 مثالًا؛ وكل شريحة أصغر تحمل `SMALL_SLICE`. لا نخفيها ولا نعمم منها، بل نستخدمها لإنتاج سؤال أو خطة جمع بيانات.


In [ ]:
def sliced_report(rows, pred_key, slice_keys, min_slice_size=15, n_boot=500):
    groups = [("ALL", rows)]
    for key in slice_keys:
        for value in sorted({row[key] for row in rows}):
            groups.append((f"{key}={value}", [row for row in rows if row[key] == value]))
    report = []
    for offset, (name, group) in enumerate(groups):
        interval = bootstrap_ci(
            [row["topic"] for row in group],
            [row[pred_key] for row in group],
            macro_f1,
            n_boot=n_boot,
            seed=42 + offset,
        )
        report.append({
            "slice": name,
            "n": len(group),
            "flag": "SMALL_SLICE" if len(group) < min_slice_size else "",
            **interval,
        })
    return report

slice_report = sliced_report(rows, "prediction_b", ["language", "variant", "length_bucket"])
for item in slice_report:
    print(item)


## 4) Behavioural tests: هل يحترم العقد؟

الـmetric على dataset تجيب “كم؟”. الاختبار السلوكي يجيب “هل ينجح في سلوك محدد مهم؟”. هذه الحالات مأخوذة من `COURSE_FIXTURE`; أضف حالاتك أنت لكل إصلاح قبل اعتماده.


In [ ]:
by_id = {row["example_id"]: row for row in rows}
BEHAVIOURAL_CASES = [
    {"case": "Gulf verification code → digital_service", "example_id": "EV-001", "expected": "digital_service"},
    {"case": "MSA upload failure → digital_service", "example_id": "EV-017", "expected": "digital_service"},
    {"case": "Gulf missing health result → health", "example_id": "EV-006", "expected": "health"},
    {"case": "English missing lab result → health", "example_id": "EV-030", "expected": "health"},
    {"case": "MSA bus route → transport", "example_id": "EV-020", "expected": "transport"},
    {"case": "English double charge → permit", "example_id": "EV-035", "expected": "permit"},
]
behavioural_results = []
for case in BEHAVIOURAL_CASES:
    actual = by_id[case["example_id"]]["prediction_b"]
    behavioural_results.append({**case, "actual": actual, "passed": actual == case["expected"]})
behavioural_summary = {
    "passed": sum(row["passed"] for row in behavioural_results),
    "total": len(behavioural_results),
}
behavioural_summary["pass_rate"] = behavioural_summary["passed"] / behavioural_summary["total"]
for row in behavioural_results:
    print(row)
print("COURSE_FIXTURE behavioural", behavioural_summary)


## 5) Taxonomy يدوية، لا أوتوماتيكية

اقرأ النص والتوقع والlabel والسياق. اختر tag واحدة أولية، واكتب rationale قصيرًا. القائمة التالية مثال تعليمي على أخطاء B؛ استبدلها بأخطاء validation في مشروعك.

Tags: `label_noise`, `class_confusion`, `dialect_gap`, `negation`, `truncation`, `preprocessing`, `entity_boundary`, `hard_or_ambiguous`.


In [ ]:
ALLOWED_TAGS = {
    "label_noise", "class_confusion", "dialect_gap", "negation",
    "truncation", "preprocessing", "entity_boundary", "hard_or_ambiguous",
}
TAGGED_ERRORS = [
    {"example_id": "EV-006", "taxonomy_tag": "dialect_gap", "rationale": "Gulf phrasing for a missing health result confused with transport."},
    {"example_id": "EV-007", "taxonomy_tag": "hard_or_ambiguous", "rationale": "The attachment mention lacks an explicit permit cue."},
    {"example_id": "EV-010", "taxonomy_tag": "dialect_gap", "rationale": "Gulf app wording obscures the prescription-renewal intent."},
    {"example_id": "EV-012", "taxonomy_tag": "dialect_gap", "rationale": "Colloquial location question lacks the formal transport terms."},
    {"example_id": "EV-019", "taxonomy_tag": "class_confusion", "rationale": "Status language overlaps generic digital-service failures."},
    {"example_id": "EV-020", "taxonomy_tag": "class_confusion", "rationale": "Map/application cue dominates the route intent."},
    {"example_id": "EV-030", "taxonomy_tag": "hard_or_ambiguous", "rationale": "Missing-record wording overlaps multiple service domains."},
    {"example_id": "EV-031", "taxonomy_tag": "hard_or_ambiguous", "rationale": "The short request omits the permit noun."},
]

seen = set()
for item in TAGGED_ERRORS:
    assert item["taxonomy_tag"] in ALLOWED_TAGS
    assert item["example_id"] not in seen
    source_row = by_id[item["example_id"]]
    assert source_row["prediction_b"] != source_row["topic"]
    seen.add(item["example_id"])
taxonomy_counts = Counter(item["taxonomy_tag"] for item in TAGGED_ERRORS)
print("COURSE_FIXTURE taxonomy", dict(taxonomy_counts))


## 6) من الخطأ إلى خطة إصلاح | Evidence → action

صيغة الإصلاح الجيد: **المشكلة + الدليل + التغيير + اختبار القبول**.

| الأولوية | إصلاح fixture المقترح | دليل البداية | اختبار القبول |
|---:|---|---|---|
| 1 | زيادة وتنقيح أمثلة Gulf للصحة والنقل | `dialect_gap` متكرر | behavioural cases الجديدة + slice CI |
| 2 | إضافة أمثلة contrastive لطلب/تطبيق مقابل تصريح/مسار | `class_confusion` | انخفاض أخطاء الزوج دون إضرار macro-F1 |
| 3 | مراجعة الإرشادات وجمع سياق للأمثلة القصيرة الملتبسة | `hard_or_ambiguous` | agreement review ثم metric جديد |

هذه ليست وصفة إنتاج؛ هي قراءة قابلة للاختبار من fixture. في مشروعك يجب أن تأتي الأولويات من أخطائك أنت.


In [ ]:
recommendations = [
    {
        "priority": 1,
        "issue": "Gulf coverage for health and transport",
        "evidence": "dialect_gap tags in validation fixture",
        "change": "collect/review targeted Gulf examples",
        "acceptance_test": "new behavioural cases plus sliced CI",
    },
    {
        "priority": 2,
        "issue": "class confusion around app/status wording",
        "evidence": "EV-019 and EV-020",
        "change": "add contrastive examples and review label guide",
        "acceptance_test": "paired comparison without regression in other slices",
    },
    {
        "priority": 3,
        "issue": "underspecified short requests",
        "evidence": "hard_or_ambiguous tags",
        "change": "request context or abstain when confidence is low",
        "acceptance_test": "ambiguity behavioural suite",
    },
]
print(json.dumps(recommendations, ensure_ascii=False, indent=2))


## 7) حفظ التقرير | Save inspectable evidence

ملفات JSON/CSV التالية أدلة وسيطة. انسخ النتائج المهمة إلى `EVALUATION_REPORT.md` و`MODEL_CARD.md` مع وصف البيانات والقيود والقرار.


In [ ]:
reports_dir = Path("reports")
reports_dir.mkdir(exist_ok=True)
evaluation_report = {
    "data_kind": DATA_KIND,
    "split": "validation",
    "rows": len(rows),
    "macro_f1_a": interval_a,
    "macro_f1_b": interval_b,
    "paired_b_minus_a": paired_result,
    "paired_verdict": verdict,
    "behavioural": behavioural_summary,
    "taxonomy_counts": dict(taxonomy_counts),
    "top_fixes": recommendations,
}
(reports_dir / "day3_evaluation_fixture.json").write_text(
    json.dumps(evaluation_report, ensure_ascii=False, indent=2), encoding="utf-8"
)

with (reports_dir / "day3_slice_report.csv").open("w", encoding="utf-8", newline="") as handle:
    writer = csv.DictWriter(handle, fieldnames=list(slice_report[0]))
    writer.writeheader()
    writer.writerows(slice_report)
with (reports_dir / "day3_error_taxonomy.csv").open("w", encoding="utf-8", newline="") as handle:
    writer = csv.DictWriter(handle, fieldnames=list(TAGGED_ERRORS[0]))
    writer.writeheader()
    writer.writerows(TAGGED_ERRORS)
print("REPORTS_WRITTEN", sorted(path.name for path in reports_dir.glob("day3_*")))


## بوابة Core | Core gate

علامة النجاح تتحقق من provenance، CI، المقارنة الزوجية، الشرائح، taxonomy، والتقارير. النجاح البرمجي لا يعوض كتابة تفسيرك في تقرير المشروع.


In [ ]:
core_checks = {
    "fixture_disclosed": evaluation_report["data_kind"] == "COURSE_FIXTURE",
    "validation_only": evaluation_report["split"] == "validation",
    "confidence_intervals": interval_a["ci_low"] <= interval_a["estimate"] <= interval_a["ci_high"],
    "paired_comparison": "supports_directional_claim" in paired_result,
    "slices_present": len(slice_report) >= 8,
    "small_slice_flags_present": any(row["flag"] == "SMALL_SLICE" for row in slice_report),
    "behavioural_tests": behavioural_summary["total"] >= 5,
    "manual_taxonomy": len(TAGGED_ERRORS) >= 5 and bool(taxonomy_counts),
    "three_ranked_fixes": [row["priority"] for row in recommendations] == [1, 2, 3],
    "reports_written": all((reports_dir / name).exists() for name in [
        "day3_evaluation_fixture.json", "day3_slice_report.csv", "day3_error_taxonomy.csv"
    ]),
}
assert all(core_checks.values()), core_checks
print(core_checks)
print("DAY3_NOTEBOOK7_CORE=PASS")


## بعد المختبر | After the lab

1. استبدل fixture بتنبؤات مشروعك على validation.
2. أكمل [قالب تقرير التقييم](../templates/EVALUATION_REPORT_TEMPLATE.md) و[بطاقة النموذج](../templates/MODEL_CARD_TEMPLATE.md).
3. اكتب تفسير CI والشرائح وثلاثة إصلاحات في `DECISIONS.md`.
4. استخدم commit: `docs: add evaluation and error analysis`.
5. ارجع إلى [Gate C](../day-03/04-labs-checkpoint.md) وأكمل checklist قبل مغادرة اليوم الثالث.
